# ContractRisk M3 — RAGAS 1.1 sobre las respuestas guardadas

Recalcula RAGAS a partir de las respuestas ya guardadas de la evaluación final (`m3/results/final_m3/harness_rows.jsonl`). No regenera respuestas ni juicios del harness. Versión 1.1:

- Segmentación de afirmaciones corregida: la versión 1.0 partía `CO1.PCCNTR.<n>` en sus puntos.
- Variante **exacta del laboratorio S10**: context precision es la fracción de pasajes relevantes para la pregunta, y context recall es un sí/no sobre la referencia completa.
- **Adaptación** de la versión 1.0: precisión ponderada por rango y recall por segmentos.
- Ambas variantes se reportan por separado.

Juez: Qwen2.5-3B-Instruct local, el mismo modelo del generador (dependencia declarada en el reporte).

## Cómo ejecutar

- **Requisitos:** Colab con GPU T4, en una sesión nueva, sin claves de pago.
- **Orden:** ejecutar las celdas de arriba hacia abajo. Las celdas hacen, en orden:
  1. obtienen el código;
  2. instalan las dependencias fijadas y corren las pruebas;
  3. recalculan RAGAS;
  4. muestran la tabla;
  5. empaquetan los resultados.
- **Versiones:** juez y embeddings con revisión fija, registrados en `metadata.json` junto con el hash del archivo de respuestas evaluado.
- **Tiempo estimado de ejecución:** unos 10 minutos. Al finalizar, se genera un ZIP con los resultados y metadatos de la reevaluación.

**Nota metodológica:** las respuestas evaluadas no se regeneran ni se editan, y las respuestas originales del juez se conservan, incluidas las que no se pudieron interpretar, para asegurar la trazabilidad.

In [ ]:
import os, subprocess, sys, tempfile, zipfile
from pathlib import Path
# Obtiene el código: clona la rama de entrega (USE_ZIP = False) o usa un paquete
# generado con `git archive` (contractrisk_m3_ragas_v1_1_codigo.zip).
USE_ZIP = False
REPO = 'https://github.com/vcastrop/Juan-Camilo-Ramon-Valentina-Castro.git'
BRANCH = 'Entrega/contractrisk-m3'
ROOT = Path('/content/contractrisk_m3_ragas_v1_1')
os.chdir('/content')  # el directorio actual puede no existir si la celda se re-ejecutó tras borrar ROOT
if ROOT.exists():
    raise RuntimeError('Use una sesión nueva: la carpeta ya existe y la evidencia anterior no se sobrescribe')
if USE_ZIP:
    from google.colab import files
    uploaded = files.upload()  # se guarda en /content
    if len(uploaded) != 1:
        raise ValueError('Suba únicamente contractrisk_m3_ragas_v1_1_codigo.zip')
    ROOT.mkdir(parents=True)
    with zipfile.ZipFile(Path('/content') / next(iter(uploaded))) as z:
        for member in z.infolist():
            if not (ROOT / member.filename).resolve().is_relative_to(ROOT.resolve()):
                raise ValueError('Ruta no permitida en ZIP')
        z.extractall(ROOT)
    COMMIT = z.comment.decode() or 'zip-sin-commit'
else:
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', BRANCH, REPO, str(ROOT)], check=True)
    COMMIT = subprocess.run(['git', '-C', str(ROOT), 'rev-parse', 'HEAD'], capture_output=True, text=True, check=True).stdout.strip()
os.chdir(ROOT)
print('Código evaluado:', COMMIT)
assert Path('m3/rerun_ragas_m3.py').is_file(), 'El código no contiene la corrida final: use la rama o ZIP correctos'

In [ ]:
def run_visible(command, log_path):
    with open(log_path, 'w', encoding='utf-8') as log:
        proc = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in proc.stdout:
            print(line, end='', flush=True)
            log.write(line)
        code = proc.wait()
    if code:
        raise RuntimeError(f'Comando falló ({code}); consulte {log_path}')

run_visible([sys.executable, '-m', 'pip', 'install', '-r', 'm3/requirements-development-v2.txt'], 'installation.log')
run_visible([sys.executable, '-m', 'unittest', 'discover', '-s', 'm3/tests', '-v'], 'tests.log')
run_visible([sys.executable, '-c', "import sys; sys.path.insert(0,'m3'); from sqlite_support import ensure_sqlite; print(ensure_sqlite()); import torch; assert torch.cuda.is_available(), 'Active GPU T4'; print(torch.cuda.get_device_name(0))"], 'environment.log')

In [ ]:
run_visible([sys.executable, '-u', 'm3/rerun_ragas_m3.py', '--run', 'm3/results/final_m3', '--output', 'ragas_v1_1_resultados'], 'execution.log')

In [ ]:
from IPython.display import Markdown, display
display(Markdown(Path('ragas_v1_1_resultados/tablas_ragas_v1_1.md').read_text(encoding='utf-8')))

In [ ]:
# Empaqueta resultados, logs y commit evaluado en un ZIP descargable.
import shutil
from google.colab import files
result_dir = ROOT / 'ragas_v1_1_resultados'
result_dir.mkdir(exist_ok=True)
(result_dir / 'commit.txt').write_text(COMMIT + '\n', encoding='utf-8')
for log in ROOT.glob('*.log'):
    shutil.copy2(log, result_dir / log.name)
meta = result_dir / 'metadata.json'
print(meta.read_text(encoding='utf-8')[:3000] if meta.exists() else 'No se creó metadata: revisar logs.')
bundle = shutil.make_archive(str(ROOT / 'contractrisk_m3_ragas_v1_1_resultados'), 'zip', result_dir)
files.download(bundle)

## Resultados

La carpeta `ragas_v1_1_resultados` contiene los juicios por caso con sus respuestas originales del juez (`ragas_v1_1.jsonl`), el resumen (`ragas_m3_v1_1.csv`), la tabla (`tablas_ragas_v1_1.md`) y los metadatos. La última celda la empaqueta, con los logs y el commit evaluado, en `contractrisk_m3_ragas_v1_1_resultados.zip`. La reevaluación usada en el reporte está en `m3/results/final_m3_ragas_v1_1/`.